# Pipeline RAG untuk Literature Gap Matrix
**Google Colab + Gemini API + PDF artikel ilmiah**

Notebook ini mengerjakan ekstraksi PDF, chunking, embedding, retrieval, generation dengan Gemini, ekspor Excel, dan sintesis research gap.

**Output:** sheet `Literature Gap Matrix` dengan tepat 8 kolom: Paper, DOI, Problem, Metode, Dataset, Temuan, Keterbatasan, Future work. Nilai hasil AI diminta menyertakan bukti halaman, misalnya `[paper_01.pdf, hlm. 4]`.

> Hasil AI adalah draf. Validasi seluruh klaim dan nomor halaman pada PDF asli sebelum dikumpulkan. PDF hasil scan mungkin membutuhkan OCR.

## 1. Instalasi pustaka
Jalankan sel ini terlebih dahulu.

In [1]:
%pip -q install pymupdf sentence-transformers faiss-cpu pandas openpyxl google-genai

## 2. Impor dan konfigurasi

In [2]:
import re, json, time, getpass
from pathlib import Path
import fitz
import numpy as np
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer
from google import genai
from google.genai import types
from google.colab import files

GEMINI_MODEL = "gemini-3.8-flash"
EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
OUTPUT_DIR = Path("/content/output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

/usr/local/lib/python3.13/dist-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


## 3. Masukkan Gemini API key
Dapatkan API key dari Google AI Studio. Input di bawah tersembunyi; jangan commit key ke GitHub.

In [3]:
api_key = getpass.getpass("Masukkan Gemini API key: ").strip()
if not api_key:
    raise ValueError("API key belum diisi.")
client = genai.Client(api_key=api_key)
print("Klien Gemini siap.")

Masukkan Gemini API key: ··········
Klien Gemini siap.


## 4. Unggah artikel PDF
Mulai dengan 3–5 artikel untuk pengujian, lalu tambah artikel lainnya.

In [4]:
uploaded = files.upload()
pdf_paths = []
for filename, content in uploaded.items():
    if not filename.lower().endswith(".pdf"):
        print("Diabaikan (bukan PDF):", filename)
        continue
    safe_name = re.sub(r"[^A-Za-z0-9._-]+", "_", Path(filename).name)
    path = Path("/content") / safe_name
    path.write_bytes(content)
    pdf_paths.append(path)
if not pdf_paths:
    raise ValueError("Tidak ada PDF yang diunggah.")
print("Jumlah PDF:", len(pdf_paths))

Saving AI and Semantic Modeling for Profile–Opportunity Matching A.pdf to AI and Semantic Modeling for Profile–Opportunity Matching A (1).pdf
Jumlah PDF: 1


## 5. Ekstraksi teks per halaman
Nomor halaman disimpan untuk membantu penelusuran bukti.

In [5]:
def extract_pdf_pages(pdf_path):
    result = []
    with fitz.open(pdf_path) as pdf:
        for page_idx, page in enumerate(pdf):
            text = page.get_text("text").strip()
            if text:
                result.append({
                    "paper": pdf_path.name,
                    "page": page_idx + 1,
                    "text": text
                })
    return result

all_pages = []
pdf_stats = []
for path in pdf_paths:
    pages = extract_pdf_pages(path)
    pdf_stats.append({
        "Paper": path.name,
        "Halaman dengan teks": len(pages),
        "Jumlah karakter": sum(len(p["text"]) for p in pages)
    })
    all_pages.extend(pages)

display(pd.DataFrame(pdf_stats))
if not all_pages:
    raise ValueError("Teks tidak terbaca. PDF mungkin hasil scan dan memerlukan OCR.")

,Paper,Halaman dengan teks,Jumlah karakter
0,AI_and_Semantic_Modeling_for_Profile_Opportuni...,14,83544


## 6. Chunking
Teks dipecah menjadi bagian kecil dengan overlap, sambil mempertahankan nama artikel dan halaman.

In [6]:
def make_chunks(pages, chunk_size=1400, overlap=250):
    chunks = []
    step = chunk_size - overlap
    for page in pages:
        text = re.sub(r"\\s+", " ", page["text"]).strip()
        for start in range(0, len(text), step):
            piece = text[start:start + chunk_size].strip()
            if len(piece) >= 80:
                chunks.append({
                    "paper": page["paper"],
                    "page": page["page"],
                    "text": piece
                })
    return chunks

chunks = make_chunks(all_pages)
print("Jumlah chunk:", len(chunks))
print(chunks[0]["paper"], "halaman", chunks[0]["page"])
print(chunks[0]["text"][:400])

Jumlah chunk: 79
AI_and_Semantic_Modeling_for_Profile_Opportunity_Matching_A_1_.pdf halaman 1
AI and Semantic Modeling for Profile–Opportunity Matching: A
Systematic Review with Quantitative Synthesis
Jáder Felipe Silva Duarte
jader_duarte@live.com
d2026100837@unifei.edu.br
Federal University of Itajubá (UNIFEI)
Itabira, Minas Gerais, Brazil
Adler Diniz de Souza
adlerdiniz@unifei.edu.br
Federal University of Itajubá (UNIFEI)
Itajubá, Minas Gerais, Brazil
ABSTRACT
Digital recruitment platfo


## 7. Embedding dan indeks FAISS
Model embedding mengubah teks menjadi vektor untuk pencarian semantik.

In [7]:
embedding_model = SentenceTransformer(EMBEDDING_MODEL)
chunk_texts = [c["text"] for c in chunks]
vectors = embedding_model.encode(
    chunk_texts, normalize_embeddings=True, show_progress_bar=True
).astype("float32")

vector_index = faiss.IndexFlatIP(vectors.shape[1])
vector_index.add(vectors)
print("Vektor terindeks:", vector_index.ntotal)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Vektor terindeks: 79


## 8. Retrieval
Fungsi ini mencari chunk relevan dari artikel yang sedang dianalisis.

In [8]:
def retrieve(question, paper, top_k=5):
    eligible = [i for i, c in enumerate(chunks) if c["paper"] == paper]
    if not eligible:
        return []
    qvec = embedding_model.encode(
        [question], normalize_embeddings=True
    ).astype("float32")
    sub_vectors = vectors[eligible]
    sub_index = faiss.IndexFlatIP(sub_vectors.shape[1])
    sub_index.add(sub_vectors)
    scores, ids = sub_index.search(qvec, min(top_k, len(eligible)))
    results = []
    for score, local_id in zip(scores[0], ids[0]):
        idx = eligible[int(local_id)]
        c = chunks[idx]
        results.append({
            "paper": c["paper"], "page": c["page"],
            "score": float(score), "text": c["text"]
        })
    return results

print("Contoh retrieval:")
for r in retrieve("metode, dataset, hasil, keterbatasan, future work", pdf_paths[0].name):
    print(f"[{r['paper']}, hlm. {r['page']}] skor={r['score']:.3f}")
    print(r["text"][:250], "\\n")

Contoh retrieval:
[AI_and_Semantic_Modeling_for_Profile_Opportunity_Matching_A_1_.pdf, hlm. 2] skor=0.583
opportunities.
3.2
Research Questions and PICOC
The review was guided by six research questions formulated
to connect the study scope, data extraction, subgroup-based
quantitative synthesis, and discussion of methodological limi-
tations:
∙RQ1 – Tech \n
[AI_and_Semantic_Modeling_for_Profile_Opportunity_Matching_A_1_.pdf, hlm. 2] skor=0.565
r–item interac-
tions, and graph-based recommendation, showing that strong
predictive performance does not necessarily imply equivalent
ranking performance [12].
3
METHODOLOGY
This systematic review was conducted based on a protocol
defined in Parsif \n
[AI_and_Semantic_Modeling_for_Profile_Opportunity_Matching_A_1_.pdf, hlm. 1] skor=0.563
fficiently documented data reduces reproducibility and
limits external validation. Therefore, the evidence must be
organized by tasks, metric families, and methodologically
comparable subgroups.
This challenge 

## 9. Ekstraksi informasi dengan Gemini
Model diminta menghasilkan JSON untuk delapan kolom, dengan bukti halaman. Jika informasi tidak muncul di cuplikan, model harus menyatakan keterbatasan bukti.

In [9]:
COLUMNS = [
    "Paper", "DOI", "Problem", "Metode", "Dataset",
    "Temuan", "Keterbatasan", "Future work"
]

def find_doi(text):
    match = re.search(r"10\\.\\d{4,9}/[-._;()/:A-Z0-9]+", text, flags=re.I)
    return match.group(0).rstrip(".,;:)") if match else "Tidak ditemukan pada teks PDF"

def call_gemini_json(prompt, retries=6):
    last_error = None
    for attempt in range(retries):
        try:
            response = client.models.generate_content(
                model=GEMINI_MODEL,
                contents=prompt,
                config=types.GenerateContentConfig(
                    temperature=0.1,
                    response_mime_type="application/json"
                )
            )
            return json.loads(response.text)
        except Exception as exc:
            last_error = exc
            if attempt < retries - 1:
                time.sleep(2 * (attempt + 1))
    raise RuntimeError(f"Gemini gagal: {last_error}")

def analyze_one_paper(paper):
    questions = [
        "judul, DOI, tujuan, problem penelitian",
        "metode penelitian, algoritma, pendekatan",
        "dataset, jumlah sampel, sumber data, periode",
        "hasil utama, temuan, metrik evaluasi",
        "keterbatasan yang dinyatakan penulis",
        "future work atau rekomendasi penelitian lanjutan"
    ]
    evidence = []
    seen = set()
    for question in questions:
        for item in retrieve(question, paper, top_k=3):
            key = (item["page"], item["text"])
            if key not in seen:
                seen.add(key)
                evidence.append(item)

    evidence_text = "\\n\\n".join(
        f"[SUMBER: {x['paper']}, hlm. {x['page']}]\\n{x['text']}"
        for x in evidence
    )
    full_text = next((p["text"] for p in all_pages if p["paper"] == paper), "")
    doi_hint = find_doi(full_text)

    prompt = (
        "Anda asisten riset akademik. Analisis HANYA konteks artikel berikut.\\n"
        f"NAMA FILE: {paper}\\nDOI hasil pencarian awal (verifikasi): {doi_hint}\\n"
        f"Buat satu objek JSON valid dengan kunci persis: {COLUMNS}.\\n"
        "Aturan: Bahasa Indonesia akademik dan ringkas. Problem = masalah/tujuan; "
        "Metode = desain/algoritma; Dataset = sumber/ukuran/periode jika tersedia; "
        "Temuan = hasil yang benar-benar dilaporkan; Keterbatasan = limitation yang "
        "dinyatakan penulis; Future work = arah lanjutan yang dinyatakan penulis. "
        "Jika tidak ditemukan dalam cuplikan, tulis 'Tidak ditemukan dalam cuplikan yang diambil'. "
        "Jangan mengarang angka, DOI, atau halaman. Setiap nilai selain Paper harus "
        "menyertakan bukti halaman jika didukung, format [nama_file.pdf, hlm. N]. "
        "Bedakan fakta dan ketidakpastian. Jangan keluarkan markdown atau teks di luar JSON.\\n"
        f"Nilai Paper harus persis: {paper}.\\n\\nKONTEKS ARTIKEL:\\n{evidence_text}"
    )
    result = call_gemini_json(prompt)
    row = {col: str(result.get(col, "Tidak ditemukan dalam cuplikan yang diambil")).strip()
           for col in COLUMNS}
    row["Paper"] = paper
    return row

preview_row = analyze_one_paper(pdf_paths[0].name)
display(pd.DataFrame([preview_row], columns=COLUMNS))

,Paper,DOI,Problem,Metode,Dataset,Temuan,Keterbatasan,Future work
0,AI_and_Semantic_Modeling_for_Profile_Opportuni...,Tidak ditemukan dalam cuplikan yang diambil,Literatur mengenai pencocokan profil dengan pe...,Tinjauan literatur sistematis menggunakan kera...,Korpus terdiri dari 43 studi yang terpilih dar...,Tren metodologis beralih dari tumpang-tindih l...,"Proses seleksi studi, penapisan teks lengkap, ...",Diperlukan peningkatan dalam aspek reproduksib...


## 10. Analisis semua artikel
Periksa hasil uji satu artikel terlebih dahulu. Panggilan Gemini dapat memakai kuota atau menimbulkan biaya sesuai akun/model.

In [10]:
all_rows = []
for i, path in enumerate(pdf_paths, start=1):
    print(f"[{i}/{len(pdf_paths)}] {path.name}")
    try:
        all_rows.append(analyze_one_paper(path.name))
    except Exception as exc:
        print("Gagal:", exc)
        all_rows.append({
            "Paper": path.name,
            "DOI": find_doi(next((p["text"] for p in all_pages if p["paper"] == path.name), "")),
            "Problem": "GAGAL DIANALISIS; periksa error",
            "Metode": "GAGAL DIANALISIS",
            "Dataset": "GAGAL DIANALISIS",
            "Temuan": "GAGAL DIANALISIS",
            "Keterbatasan": "GAGAL DIANALISIS",
            "Future work": "GAGAL DIANALISIS"
        })

matrix_df = pd.DataFrame(all_rows, columns=COLUMNS)
display(matrix_df)

[1/1] AI_and_Semantic_Modeling_for_Profile_Opportunity_Matching_A_1_.pdf


,Paper,DOI,Problem,Metode,Dataset,Temuan,Keterbatasan,Future work
0,AI_and_Semantic_Modeling_for_Profile_Opportuni...,Tidak ditemukan dalam cuplikan yang diambil,Literatur pencocokan profil-peluang (profile–o...,Tinjauan pustaka sistematis (systematic litera...,Sebanyak 43 studi akhir yang disertakan dalam ...,Terdapat pergeseran dari pencocokan leksikal m...,"Proses seleksi studi, penyaringan teks lengkap...",Diperlukan kemajuan dalam standardisasi protok...


## 11. Sintesis research gap lintas artikel
Sintesis ini adalah interpretasi awal, bukan bukti otomatis bahwa novelty sudah terbukti.

In [11]:
def synthesize_research_gap(matrix):
    matrix_json = json.dumps(
        matrix.fillna("").to_dict(orient="records"),
        ensure_ascii=False, indent=2
    )
    prompt = (
        "Anda peneliti yang menyusun sintesis literature gap berdasarkan matriks di bawah. "
        "Tulis dalam Bahasa Indonesia dengan bagian: (1) pola tema/metode, "
        "(2) perbandingan dataset/evaluasi jika tersedia, (3) keterbatasan berulang "
        "dengan nama paper, (4) kandidat research gap yang didukung, "
        "(5) future work yang diturunkan secara hati-hati, (6) hal yang perlu diverifikasi. "
        "Sertakan nama paper untuk setiap klaim. Bedakan fakta dari interpretasi. "
        "Jangan mengklaim novelty telah terbukti. Jika bukti kurang, nyatakan keterbatasannya.\\n\\n"
        "MATRIKS:\\n" + matrix_json
    )
    response = client.models.generate_content(
        model=GEMINI_MODEL,
        contents=prompt,
        config=types.GenerateContentConfig(temperature=0.2)
    )
    return response.text

gap_analysis = synthesize_research_gap(matrix_df)
print(gap_analysis)

Berikut adalah sintesis *literature gap* berdasarkan matriks yang disediakan. Perlu dicatat bahwa matriks ini **hanya memuat satu dokumen** (sebuah *systematic literature review* / SLR), sehingga sintesis di bawah ini didasarkan sepenuhnya pada temuan sekunder dari dokumen tunggal tersebut dan bukan hasil komparasi lintas-studi mandiri.

---

### 1. Pola Tema dan Metode
* **Fakta:** Berdasarkan tinjauan pustaka sistematis dalam `AI_and_Semantic_Modeling_for_Profile_Opportunity_Matching_A_1_.pdf`, literatur *profile–opportunity matching* menunjukkan pergeseran metodologis dari pencocokan leksikal konvensional menuju representasi semantik. Rumpun metode yang teridentifikasi dominan mencakup model berbasis Transformer, sistem rekomendasi (*recommender systems*), pendekatan graf (*Graph Neural Networks* / GNN), dan ontologi/pengetahuan terstruktur. Tugas (*task*) analitis yang paling sering diformulasikan adalah klasifikasi resume-pekerjaan, pemeringkatan (*ranking*), dan prediksi kesesuai

## 12. Ekspor Excel
Sheet pertama memiliki tepat 8 kolom. Sheet kedua berisi sintesis gap; sheet ketiga statistik ekstraksi.

In [12]:
output_file = OUTPUT_DIR / "literature_gap_matrix_rag.xlsx"
with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    matrix_df.to_excel(writer, sheet_name="Literature Gap Matrix", index=False)
    pd.DataFrame({"Analisis Research Gap": [gap_analysis]}).to_excel(
        writer, sheet_name="Analisis Research Gap", index=False
    )
    pd.DataFrame(pdf_stats).to_excel(writer, sheet_name="Statistik PDF", index=False)

print("File tersimpan:", output_file)
files.download(str(output_file))

File tersimpan: /content/output/literature_gap_matrix_rag.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 13. Checklist validasi
- [ ] DOI sesuai dengan halaman artikel atau metadata penerbit.
- [ ] Problem, metode, dataset, dan temuan telah diperiksa pada PDF.
- [ ] Nomor halaman pada setiap bukti benar dan mendukung klaim.
- [ ] Keterbatasan/future work tidak dibuat-buat.
- [ ] PDF scan diproses OCR bila teks tidak terbaca.
- [ ] Research gap merupakan sintesis lintas artikel.
- [ ] API key tidak disimpan dalam notebook/repo publik.
- [ ] Hasil utama berisi delapan kolom sesuai instruksi dosen.

## Batasan
Retrieval bergantung pada kualitas PDF, chunking, dan embedding. Regex DOI hanya petunjuk awal. Bukti halaman yang ditulis AI wajib diverifikasi. Untuk artikel sangat panjang atau tabel kompleks, perlu evaluasi tambahan.